# **🧪 Experimentación y Modelado**

Registro de los experimentos del modelo final. Cada configuración se evalúa con la misma **CV estratificada de 5 folds** del baseline y se registra en **MLflow** (parámetros, métricas, lista de features y curva de ganancia). La lógica vive en `src/` y este notebook solo orquesta los experimentos.

Orden de los experimentos:

1. **Baseline replicado**, como referencia.
2. **Conjuntos de features** del notebook 03 con LightGBM por defecto.
3. **Otros modelos** con el mejor conjunto de features.
4. **Búsqueda de hiperparámetros** del mejor modelo con Optuna.
5. **Entrenamiento sensible al monto**, con pesos por transacción.
6. **Jev**, un modelo de evaluación externo, sobre una muestra.

La métrica principal sigue siendo el **% de la ganancia máxima** (+25% del monto por legítima aprobada, −100% por fraude aprobado), con el umbral que maximiza la ganancia out-of-fold. Como todos los experimentos usan los mismos folds, se comparan **fold a fold** contra el baseline.

Para explorar los runs: `mlflow ui --backend-store-uri sqlite:///mlflow.db` desde la raíz del proyecto.

## **📚 Librerías**

In [ ]:
# Librerías para manipulación de datos
import pandas as pd
import numpy as np

# Registro de experimentos y búsqueda de hiperparámetros
import mlflow
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

# Calibración de probabilidades y métricas de ranking
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, roc_auc_score

# Librerías de visualización
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Módulos del proyecto: features, validación y registro en MLflow
from src.features import COLUMNAS_ORIGINALES, cargar_datos, construir_features
from src.ganancia import MARGEN_GANANCIA, curva_ganancia, metricas_decision, umbral_optimo
from src.jev import evaluar_transacciones
from src.modelos import SEMILLA
from src.tracking import RAIZ_PROYECTO, configurar_mlflow, ejecutar_experimento
from src.validacion import crear_folds

In [ ]:
# Gráficos interactivos en el editor y PNG estático para GitHub y el informe
pio.renderers.default = 'plotly_mimetype+png'
pio.templates.default = 'plotly_white'

COLOR_PRINCIPAL = '#2a78d6'
COLOR_REFERENCIA = '#898781'
NOMBRE_EXPERIMENTO = 'fraude_shipping'

# Búsqueda de hiperparámetros: número de pruebas y semilla de los folds nuevos para validar el resultado
NUMERO_PRUEBAS = 50
SEMILLA_VALIDACION = 7

# Muestra evaluada con Jev y carpeta donde se guardan sus respuestas
TAMANO_MUESTRA_JEV = 1500
CARPETA_JEV = RAIZ_PROYECTO / 'data' / 'interim' / 'jev'
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# Compara cada experimento contra el de referencia, fold a fold, sobre los mismos folds


def comparar_experimentos(resultados, referencia):
    """Tabla con las métricas medias de cada experimento, la diferencia en ganancia y en cuántos folds supera a la referencia."""
    ganancia_referencia = resultados[referencia].metricas_por_fold['ganancia_pct_maxima']
    filas = []
    for nombre, resultado in resultados.items():
        ganancia = resultado.metricas_por_fold['ganancia_pct_maxima']
        filas.append({
            'experimento': nombre,
            'ganancia_pct_maxima': ganancia.mean(),
            'desvio': ganancia.std(),
            'diferencia_vs_referencia': (ganancia - ganancia_referencia).mean(),
            'folds_mejores': int((ganancia > ganancia_referencia).sum()),
            'auc_roc': resultado.metricas_por_fold['auc_roc'].mean(),
            'auc_pr': resultado.metricas_por_fold['auc_pr'].mean(),
            'umbral': resultado.umbral,
        })
    return pd.DataFrame(filas).set_index('experimento').round(3)


def graficar_ganancia(comparacion, referencia, titulo):
    """Ganancia media por experimento con su desvío entre folds; la referencia en gris."""
    tabla = comparacion.reset_index().sort_values('ganancia_pct_maxima')
    colores = {nombre: COLOR_REFERENCIA if nombre == referencia else COLOR_PRINCIPAL for nombre in tabla['experimento']}
    fig = px.bar(
        tabla,
        x='ganancia_pct_maxima',
        y='experimento',
        error_x='desvio',
        color='experimento',
        color_discrete_map=colores,
        orientation='h',
        text_auto='.2f',
        labels={'ganancia_pct_maxima': '% de la ganancia máxima', 'experimento': 'Experimento'},
        title=titulo,
    )
    fig.update_traces(textposition='inside', insidetextanchor='start')
    fig.update_xaxes(range=[min(70, tabla['ganancia_pct_maxima'].min() - 3), tabla['ganancia_pct_maxima'].max() + 3])
    fig.update_layout(showlegend=False, height=120 + 40 * len(tabla))
    return fig

## **🔢 Datos**

In [ ]:
# Dataset con las features del notebook 03 (la tasa de fraude de j se calcula dentro de cada fold)
datos = construir_features(cargar_datos('../data/raw/dataset.csv'))
features_nuevas = [columna for columna in datos.columns if columna not in [*COLUMNAS_ORIGINALES, 'fecha', 'fraude']]
features_nuevas

In [ ]:
# Base de MLflow del proyecto y experimento activo
configurar_mlflow(NOMBRE_EXPERIMENTO)

## **🎯 Baseline replicado**

Mismo LightGBM por defecto y mismas variables del notebook 02, registrado en MLflow como punto de partida. La única diferencia es que el nulo de las categóricas (`g`, `o`) ahora es una categoría explícita.

In [ ]:
# Baseline: LightGBM por defecto con las variables originales
resultados = {
    'baseline': ejecutar_experimento(
        datos, 'baseline', COLUMNAS_ORIGINALES, 'lightgbm', etiquetas={'etapa': 'baseline'}
    ),
}
pd.Series(resultados['baseline'].resumen).round(3)

**💡 Hallazgos**

- El baseline se replica: **77,7% de la ganancia máxima**, AUC-ROC de 0,872 y umbral de 0,22, igual que en el notebook 02. Tratar el nulo como categoría explícita no cambia el resultado.

## **🧱 Conjuntos de features**

Se parte del baseline y se cambia **un grupo de features por vez**, siempre con LightGBM por defecto. Luego se combinan las candidatas más prometedoras del notebook 03:

| Experimento | Cambio respecto del baseline |
|---|---|
| `mas_perfil_onp` | Agrega la combinación de `o`, `n` y `p` |
| `mas_hora` | Agrega la hora del día |
| `g_agrupado` | Reemplaza `g` por el país agrupado |
| `sin_j` | Quita `j` |
| `j_tasa_frecuencia` | Reemplaza `j` por su tasa de fraude (calculada dentro de cada fold) y su frecuencia |
| `j_cruda_mas_tasa` | Mantiene `j` y agrega su tasa y su frecuencia |
| `candidatas` | `perfil_onp`, `hora`, `g_agrupado` y `j` como tasa y frecuencia |
| `candidatas_sin_k` | Las candidatas sin `k`, que no mostró señal |
| `todas` | Todas las variables originales y todas las nuevas |
| `todas_sin_j_cruda` | Todas, pero con `j` solo como tasa y frecuencia |

In [ ]:
# Definición de cada conjunto: (features, columnas a las que se les calcula la tasa de fraude dentro del fold)
sin_j = [columna for columna in COLUMNAS_ORIGINALES if columna != 'j']
candidatas = [columna for columna in COLUMNAS_ORIGINALES if columna not in ('g', 'j')] + [
    'g_agrupado', 'j_frecuencia', 'perfil_onp', 'hora'
]

conjuntos_features = {
    'mas_perfil_onp': ([*COLUMNAS_ORIGINALES, 'perfil_onp'], []),
    'mas_hora': ([*COLUMNAS_ORIGINALES, 'hora'], []),
    'g_agrupado': ([columna if columna != 'g' else 'g_agrupado' for columna in COLUMNAS_ORIGINALES], []),
    'sin_j': (sin_j, []),
    'j_tasa_frecuencia': ([*sin_j, 'j_frecuencia'], ['j']),
    'j_cruda_mas_tasa': ([*COLUMNAS_ORIGINALES, 'j_frecuencia'], ['j']),
    'candidatas': (candidatas, ['j']),
    'candidatas_sin_k': ([columna for columna in candidatas if columna != 'k'], ['j']),
    'todas': ([*COLUMNAS_ORIGINALES, *features_nuevas], ['j']),
    'todas_sin_j_cruda': ([*sin_j, *features_nuevas], ['j']),
}

In [ ]:
# Un run de MLflow por conjunto de features
for nombre, (features, columnas_tasa) in conjuntos_features.items():
    resultados[nombre] = ejecutar_experimento(
        datos, nombre, features, 'lightgbm', columnas_tasa=columnas_tasa, etiquetas={'etapa': 'features'}
    )

In [ ]:
# Comparación fold a fold contra el baseline
comparacion_features = comparar_experimentos(resultados, 'baseline')
comparacion_features.sort_values('ganancia_pct_maxima', ascending=False)

In [ ]:
# Ganancia de cada conjunto de features
graficar_ganancia(comparacion_features, 'baseline', 'Ganancia por conjunto de features (LightGBM por defecto)').show()

**💡 Hallazgos**

- **La `j` cruda perjudica al modelo.** Con solo quitarla (`sin_j`), el AUC-ROC sube de 0,872 a 0,887. Esto confirma lo que sugerían el baseline y el notebook 03: LightGBM memoriza las 8.324 categorías en train y eso no generaliza. Mientras la `j` cruda esté presente, el AUC queda en ~0,872 aunque se agreguen su tasa y su frecuencia (`j_cruda_mas_tasa`, `todas`).
- **Reemplazar `j` por su tasa de fraude y su frecuencia** (`j_tasa_frecuencia`) mejora la ganancia en 0,75 puntos y lo hace **en los 5 folds**. Es el cambio individual más importante.
- **`candidatas` es el mejor conjunto: 79,0% de la ganancia máxima**, +1,2 puntos sobre el baseline en los 5 folds, con AUC-ROC de 0,889 y AUC-PR de 0,467 (frente a 0,436). Recupera el 42,5% de la ganancia que se pierde por fraude al aprobar todo, contra el 39,2% del baseline.
- Por separado, `perfil_onp` (+0,24), `hora` (+0,09) y `g_agrupado` (+0,09) aportan poco y no mejoran en todos los folds. A pesar de ser la variable con más señal individual en el notebook 03, `perfil_onp` casi no suma: LightGBM ya aprende la interacción entre `o`, `n` y `p` por su cuenta.
- **Las demás features nuevas no suman sobre las candidatas.** `todas_sin_j_cruda` agrega las 13 restantes (actividad reciente, flags, ratios, monto relativo, etc.) y llega a 79,0%: el mismo AUC-ROC y AUC-PR que `candidatas`, y mejor solo en 3 de los 5 folds. Lo que perjudicaba a `todas` era la `j` cruda, no el resto de las features.
- Quitar `k` no mejora (78,7% frente a 79,0%): la diferencia está dentro de la variación entre folds, así que `k` no molesta aunque no aporte.
- El umbral óptimo de `candidatas` baja a 0,16, por debajo del teórico de 0,20. Con la tasa de `j` como feature, las probabilidades del modelo parecen algo menos calibradas. Habrá que revisarlo después del tuning.
- Se elige **`candidatas`** como conjunto de features para comparar los demás modelos. Rinde igual que `todas_sin_j_cruda` con 19 variables en lugar de 33, así que es más simple de mantener y de explicar.

## **🤖 Comparación de modelos**

Con el conjunto `candidatas` fijo, se comparan cinco modelos con sus **parámetros por defecto**, para que la diferencia refleje el algoritmo y no el ajuste:

| Modelo | Manejo de categóricas y nulos |
|---|---|
| **LightGBM** | Nativo para ambos (es el run `candidatas` de la sección anterior) |
| **XGBoost** | Categóricas nativas (`enable_categorical`) y nulos nativos |
| **CatBoost** | Categóricas con *target statistics* ordenados, pensados para evitar la filtración de la etiqueta |
| **Random Forest** | Categóricas codificadas como enteros; nulos nativos de scikit-learn |
| **Regresión logística** | Imputación por mediana con indicador de nulo, normalización por cuantiles y one-hot de las categorías con al menos 100 transacciones |

La referencia de la comparación fold a fold es LightGBM.

In [ ]:
# Un run de MLflow por modelo; LightGBM reutiliza el run de candidatas
modelos_comparados = ['xgboost', 'catboost', 'random_forest', 'regresion_logistica']

resultados_modelos = {'lightgbm': resultados['candidatas']}
for nombre_modelo in modelos_comparados:
    resultados_modelos[nombre_modelo] = ejecutar_experimento(
        datos, f'candidatas_{nombre_modelo}', candidatas, nombre_modelo, columnas_tasa=['j'], etiquetas={'etapa': 'modelos'}
    )

In [ ]:
# Comparación fold a fold contra LightGBM
comparacion_modelos = comparar_experimentos(resultados_modelos, 'lightgbm')
comparacion_modelos.sort_values('ganancia_pct_maxima', ascending=False)

In [ ]:
# Ganancia de cada modelo
graficar_ganancia(comparacion_modelos, 'lightgbm', 'Ganancia por modelo (conjunto candidatas, parámetros por defecto)').show()

In [ ]:
# Curva de ganancia out-of-fold de cada modelo según el umbral, en % de la ganancia máxima
ganancia_maxima = (MARGEN_GANANCIA * datos.loc[datos['fraude'] == 0, 'monto']).sum()
curvas_modelos = pd.concat(
    [
        curva_ganancia(datos['fraude'], datos['monto'], resultado.probabilidad_oof).assign(modelo=nombre_modelo)
        for nombre_modelo, resultado in resultados_modelos.items()
    ],
    ignore_index=True,
).assign(ganancia_pct_maxima=lambda curvas: curvas['ganancia'] / ganancia_maxima * 100)

fig = px.line(
    curvas_modelos,
    x='umbral',
    y='ganancia_pct_maxima',
    color='modelo',
    labels={'umbral': 'Umbral de probabilidad de fraude', 'ganancia_pct_maxima': '% de la ganancia máxima', 'modelo': 'Modelo'},
    title='Ganancia out-of-fold según el umbral de decisión',
)
fig.update_yaxes(range=[50, curvas_modelos['ganancia_pct_maxima'].max() + 3])
fig.show()

**💡 Hallazgos**

- **LightGBM tiene la mayor ganancia (79,0%)**, pero CatBoost (78,8%) y Random Forest (78,8%) quedan a solo 0,2 puntos. Esa diferencia es menor que el desvío entre folds (~1 punto), y cada uno supera a LightGBM en 2 de los 5 folds. Los tres están **técnicamente empatados**.
- **CatBoost tiene el mejor ranking** (AUC-ROC 0,890 y AUC-PR 0,476), pero tardó **~11 minutos** por CV, frente a ~6 segundos de LightGBM: unas 100 veces más. Con ese costo, una búsqueda de hiperparámetros con decenas de pruebas no es viable en este entorno.
- **Random Forest logra la misma ganancia con un AUC bastante menor** (0,864). La ganancia no depende de ordenar bien todas las transacciones, sino de acertar en la zona de decisión y en los montos altos, así que dos modelos con distinto AUC pueden rendir parecido en el negocio.
- **XGBoost queda 1,7 puntos abajo** (77,3%) y solo supera a LightGBM en 1 fold. Con sus valores por defecto (tasa de aprendizaje de 0,3, más agresiva que la de LightGBM) probablemente sobreajusta, así que es el que más podría mejorar con ajuste.
- **La regresión logística llega a 77,1%**, casi lo mismo que el baseline de LightGBM con las variables originales (77,7%). Con buenas features, incluso un modelo lineal captura la mayor parte de la señal. Es la opción más interpretable si eso llegara a pesar más que el último punto de ganancia.
- Se elige **LightGBM** para la búsqueda de hiperparámetros: tiene la mayor ganancia, es el más rápido y permite correr muchas pruebas con Optuna.

## **🎛️ Búsqueda de hiperparámetros**

Optuna busca los hiperparámetros de LightGBM que **maximizan la ganancia out-of-fold media**, con el conjunto `candidatas` y los mismos folds. Cada prueba queda registrada en MLflow como un run anidado dentro de `optuna_lightgbm`.

- La primera prueba son los **parámetros por defecto**, así que la búsqueda parte del resultado ya conocido.
- El espacio de búsqueda cubre la complejidad de los árboles (`num_leaves`, `min_child_samples`), el ritmo de aprendizaje (`learning_rate`, `n_estimators`), el submuestreo de filas y columnas y la regularización.
- Elegir hiperparámetros con los mismos folds con los que se evalúa **sobreestima la mejora**. Por eso, al final, los parámetros elegidos se vuelven a evaluar con **folds nuevos**, generados con otra semilla.

In [ ]:
# Parámetros por defecto de LightGBM, para usarlos como primera prueba
PARAMETROS_DEFECTO = {
    'n_estimators': 100,
    'learning_rate': 0.1,
    'num_leaves': 31,
    'min_child_samples': 20,
    'subsample': 1.0,
    'colsample_bytree': 1.0,
    'reg_alpha': 0.001,
    'reg_lambda': 0.001,
}


def objetivo(prueba):
    """Ganancia out-of-fold media de LightGBM con los hiperparámetros propuestos por Optuna."""
    parametros = {
        'n_estimators': prueba.suggest_int('n_estimators', 100, 1500, log=True),
        'learning_rate': prueba.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': prueba.suggest_int('num_leaves', 8, 256, log=True),
        'min_child_samples': prueba.suggest_int('min_child_samples', 10, 500, log=True),
        'subsample': prueba.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': prueba.suggest_float('colsample_bytree', 0.3, 1.0),
        'reg_alpha': prueba.suggest_float('reg_alpha', 0.001, 10, log=True),
        'reg_lambda': prueba.suggest_float('reg_lambda', 0.001, 10, log=True),
    }
    resultado = ejecutar_experimento(
        datos, f'prueba_{prueba.number}', candidatas, 'lightgbm', parametros,
        columnas_tasa=['j'], etiquetas={'etapa': 'tuning'}, anidado=True,
    )
    return resultado.resumen['ganancia_pct_maxima_media']

In [ ]:
# Búsqueda bayesiana (TPE); todas las pruebas quedan dentro del run padre optuna_lightgbm
estudio = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEMILLA))
estudio.enqueue_trial(PARAMETROS_DEFECTO)

with mlflow.start_run(run_name='optuna_lightgbm'):
    mlflow.set_tags({'etapa': 'tuning', 'modelo': 'lightgbm'})
    estudio.optimize(objetivo, n_trials=NUMERO_PRUEBAS)
    mlflow.log_params(estudio.best_params)
    mlflow.log_metric('mejor_ganancia_pct_maxima_media', estudio.best_value)

print(f'Mejor prueba: {estudio.best_trial.number} — {estudio.best_value:.3f}% de la ganancia máxima')
estudio.best_params

In [ ]:
# Las 10 mejores pruebas
(
    estudio.trials_dataframe()
    .sort_values('value', ascending=False)
    .head(10)
    .filter(regex='^(number|value|duration|params_)')
    .rename(columns=lambda columna: columna.replace('params_', ''))
    .round(4)
)

In [ ]:
# Evolución de la búsqueda e importancia de cada hiperparámetro
plot_optimization_history(estudio).show()
plot_param_importances(estudio).show()

**💡 Hallazgos**

- La mejor prueba (la 40) llega a **79,7% de la ganancia máxima** sobre los folds de la búsqueda, 0,7 puntos más que LightGBM por defecto (79,0%). Pero 39 de las 50 pruebas superan el 79%: hay una **zona amplia de configuraciones equivalentes**, no un óptimo puntual.
- Las mejores configuraciones comparten un patrón: **tasa de aprendizaje baja** (0,012–0,022) compensada con más árboles (430–790), **hojas grandes** (`num_leaves` de 105 a 230) que exigen **muchos casos por hoja** (`min_child_samples` de 100 a 310) y casi todas las columnas en cada árbol (`colsample_bytree` de 0,82 a 0,99). Es decir, árboles más complejos que aprenden despacio, sin bajar a hojas tan chicas que permitan memorizar.
- La cantidad de árboles (41%) y la fracción de columnas por árbol (29%) explican la mayor parte de la variación de la ganancia. La regularización (`reg_alpha`, `reg_lambda`) y el submuestreo de filas casi no influyen (menos del 3% cada uno).
- ⚠️ La métrica es **ruidosa**: la prueba 0, con los parámetros por defecto más una regularización mínima (prácticamente el mismo modelo), dio 78,6%, contra 79,0% del run `candidatas`. Cambios triviales mueven la ganancia ~0,3 puntos, porque cambian el umbral elegido. Parte de la mejora de la búsqueda puede ser ese ruido, y por eso hace falta la validación con folds nuevos.

## **✅ Validación del tuning con folds nuevos**

Para medir la mejora real, LightGBM con los parámetros por defecto y con los parámetros elegidos se evalúan sobre **folds nuevos** (semilla 7), que no intervinieron en la búsqueda. También se revisa la **calibración** de las probabilidades: si el modelo está bien calibrado, su umbral óptimo debería acercarse al teórico de 0,20.

In [ ]:
# LightGBM por defecto vs. con los parámetros elegidos, sobre folds que Optuna no vio
mejores_parametros = {**estudio.best_params, 'subsample_freq': 1}
folds_nuevos = crear_folds(datos['fraude'], semilla=SEMILLA_VALIDACION)

resultados_validacion = {
    'lightgbm_defecto': ejecutar_experimento(
        datos, 'validacion_lightgbm_defecto', candidatas, 'lightgbm',
        columnas_tasa=['j'], etiquetas={'etapa': 'validacion_tuning'}, folds=folds_nuevos,
    ),
    'lightgbm_tuneado': ejecutar_experimento(
        datos, 'validacion_lightgbm_tuneado', candidatas, 'lightgbm', mejores_parametros,
        columnas_tasa=['j'], etiquetas={'etapa': 'validacion_tuning'}, folds=folds_nuevos,
    ),
}
comparar_experimentos(resultados_validacion, 'lightgbm_defecto')

In [ ]:
# Curva de calibración: probabilidad predicha media vs. tasa de fraude observada, por decil de probabilidad
fig = go.Figure()
for nombre, resultado in resultados_validacion.items():
    tasa_observada, probabilidad_media = calibration_curve(
        datos['fraude'], resultado.probabilidad_oof, n_bins=10, strategy='quantile'
    )
    fig.add_trace(go.Scatter(x=probabilidad_media, y=tasa_observada, mode='lines+markers', name=nombre))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Calibración perfecta', line={'dash': 'dash', 'color': COLOR_REFERENCIA}))
fig.update_layout(
    title='Curva de calibración (folds nuevos)',
    xaxis_title='Probabilidad predicha media',
    yaxis_title='Tasa de fraude observada',
    height=450,
)
fig.show()

In [ ]:
# Ganancia con el umbral teórico (0,20) frente al umbral óptimo, para medir cuánto cuesta no ajustar el umbral
UMBRAL_TEORICO = MARGEN_GANANCIA / (1 + MARGEN_GANANCIA)
ganancia_maxima_total = (MARGEN_GANANCIA * datos.loc[datos['fraude'] == 0, 'monto']).sum()

pd.DataFrame([
    {
        'modelo': nombre,
        'umbral_optimo': resultado.umbral,
        'ganancia_umbral_optimo': curva_ganancia(datos['fraude'], datos['monto'], resultado.probabilidad_oof, [resultado.umbral])['ganancia'].iloc[0] / ganancia_maxima_total * 100,
        'ganancia_umbral_teorico': curva_ganancia(datos['fraude'], datos['monto'], resultado.probabilidad_oof, [UMBRAL_TEORICO])['ganancia'].iloc[0] / ganancia_maxima_total * 100,
    }
    for nombre, resultado in resultados_validacion.items()
]).set_index('modelo').round(3)

**💡 Hallazgos**

- Con folds nuevos, el modelo tuneado llega a **78,9%** frente a **78,5%** del modelo por defecto: **+0,4 puntos, mejor en 4 de los 5 folds**. Cerca de la mitad de los +0,7 de la búsqueda era optimismo por elegir y evaluar con los mismos folds. La mejora real es modesta pero consistente, y también se ve en el AUC-PR (0,474 frente a 0,465).
- Con los folds nuevos, el modelo por defecto baja de 79,0% a 78,5%: la partición en sí mueve el resultado ~0,5 puntos. Las cifras de ganancia deben leerse con un margen de ±1 punto.
- **Los dos modelos están bien calibrados**: en cada decil, la probabilidad predicha media coincide con la tasa de fraude observada (en el decil más alto, 0,31 predicho frente a 0,31 observado). El modelo tuneado subestima levemente en los deciles medios (0,008 frente a 0,011), una diferencia despreciable.
- Aun así, **el umbral óptimo (0,15–0,18) queda por debajo del teórico (0,20)**. No es un problema de calibración, sino de montos: la regla `p < 0,20` supone que el monto no depende de si la transacción es fraude, pero los fraudes son más caros (monto medio de 73 frente a 42). En la franja de probabilidad 0,15–0,20, rechazar queda casi en equilibrio (se evitan 40,1 mil de pérdida por fraude y se dejan de ganar 40,5 mil), así que la curva de ganancia es plana entre ambos umbrales.
- Usar el umbral teórico en lugar del óptimo cuesta 0,2 puntos con el modelo por defecto y 0,6 con el tuneado. Una mejora posible es que la decisión considere el monto, por ejemplo entrenando con pesos proporcionales al monto o usando un umbral que dependa de él.
- **Modelo elegido: LightGBM tuneado** con el conjunto `candidatas` y un umbral de ~0,15.

## **💰 Entrenamiento sensible al monto**

El umbral óptimo queda por debajo del teórico porque los fraudes tienen montos más altos. El modelo aprende la probabilidad de fraude tratando igual a todas las transacciones, pero la ganancia pesa cada error por su monto: aprobar un fraude de 500 cuesta 100 veces más que uno de 5. Entrenar con **pesos que dependen del monto** hace que el modelo priorice acertar en las transacciones caras.

- `sin_peso`: el LightGBM tuneado de la sección anterior.
- `peso_raiz_monto`: peso proporcional a la raíz del monto. Es un punto intermedio que reduce la influencia de los montos extremos (el máximo es 3.696).
- `peso_monto`: peso proporcional al monto, que replica cómo pesa cada transacción en la ganancia.

Se usan los parámetros tuneados y los folds nuevos, para comparar contra el resultado honesto de la sección anterior. Los pesos se normalizan a media 1. Con pesos, las probabilidades dejan de estar calibradas, así que el umbral se sigue eligiendo con la curva de ganancia.

In [ ]:
# Pesos por transacción normalizados a media 1; sin_peso reutiliza el modelo tuneado de la validación
pesos_monto = {
    'peso_raiz_monto': np.sqrt(datos['monto']) / np.sqrt(datos['monto']).mean(),
    'peso_monto': datos['monto'] / datos['monto'].mean(),
}

resultados_monto = {'sin_peso': resultados_validacion['lightgbm_tuneado']}
for nombre, pesos in pesos_monto.items():
    resultados_monto[nombre] = ejecutar_experimento(
        datos, f'validacion_lightgbm_tuneado_{nombre}', candidatas, 'lightgbm', mejores_parametros,
        columnas_tasa=['j'], etiquetas={'etapa': 'peso_monto', 'peso': nombre}, folds=folds_nuevos, pesos=pesos,
    )
comparar_experimentos(resultados_monto, 'sin_peso')

In [ ]:
# Ganancia y % de rechazos por quintil de monto, con el umbral óptimo de cada modelo: dónde se gana o se pierde
quintil_monto = pd.qcut(datos['monto'], 5).rename('quintil de monto')
ganancia_si_aprueba = np.where(datos['fraude'] == 1, -datos['monto'], MARGEN_GANANCIA * datos['monto'])

filas = []
for nombre, resultado in resultados_monto.items():
    aprobada = resultado.probabilidad_oof < resultado.umbral
    por_quintil = pd.DataFrame({
        'ganancia': np.where(aprobada, ganancia_si_aprueba, 0),
        'rechazada': ~aprobada,
        'fraude_rechazado': ~aprobada & (datos['fraude'] == 1),
    }).groupby(quintil_monto, observed=True).sum()
    filas.append(por_quintil.assign(modelo=nombre))

ganancia_por_quintil = pd.concat(filas).reset_index()
ganancia_por_quintil.pivot(index='quintil de monto', columns='modelo', values=['ganancia', 'rechazada', 'fraude_rechazado']).round(0)

In [ ]:
# Curva de ganancia según el umbral para cada esquema de pesos
curvas_monto = pd.concat(
    [
        curva_ganancia(datos['fraude'], datos['monto'], resultado.probabilidad_oof).assign(modelo=nombre)
        for nombre, resultado in resultados_monto.items()
    ],
    ignore_index=True,
).assign(ganancia_pct_maxima=lambda curvas: curvas['ganancia'] / ganancia_maxima_total * 100)

fig = px.line(
    curvas_monto,
    x='umbral',
    y='ganancia_pct_maxima',
    color='modelo',
    labels={'umbral': 'Umbral de probabilidad de fraude', 'ganancia_pct_maxima': '% de la ganancia máxima', 'modelo': 'Pesos'},
    title='Ganancia out-of-fold según el umbral, con y sin pesos por monto (folds nuevos)',
)
fig.update_yaxes(range=[50, curvas_monto['ganancia_pct_maxima'].max() + 3])
fig.show()

**💡 Hallazgos**

- **Los pesos por monto no mejoran la ganancia.** Con la raíz del monto el resultado es prácticamente igual (78,8% frente a 78,9%, mejor en 3 de 5 folds, dentro del ruido). Con el monto directo empeora 0,5 puntos y baja el AUC-PR (0,459 frente a 0,474).
- **La diferencia se decide en el quintil de montos más altos**, que concentra más de la mitad de la ganancia. Ahí, el modelo con peso por monto rechaza menos transacciones (3.356 frente a 3.720) y detecta menos fraudes (1.294 frente a 1.366), así que gana menos (725 mil frente a 731 mil). En los quintiles bajos, en cambio, rechaza más.
- La explicación es que el monto va de 0,02 a 3.696: con pesos proporcionales al monto, unas pocas transacciones caras dominan el entrenamiento, y el modelo aprende con **mucha más varianza**. La raíz suaviza ese efecto, pero no alcanza para mejorar.
- Tiene sentido en teoría: `monto` ya es una feature y las probabilidades del modelo sin pesos están bien calibradas, así que ya tiene la información necesaria para decidir bien. Los pesos cambian qué errores se priorizan, pero no agregan información.
- Se mantiene el **modelo sin pesos**. El desfase del umbral (0,15 frente a 0,20) cuesta poco, porque la curva de ganancia es plana en esa zona.

## **🧠 Jev: modelo de evaluación externo**

**Jev** (TypeSafe AI, a través de Vercel AI Gateway) es un modelo de evaluación: recibe un estado y preguntas tipadas y devuelve probabilidades, sin entrenarse con nuestros datos. Evalúa cada transacción **en frío**, solo a partir de sus campos y de las instrucciones que se le dan. Se prueban dos variantes:

- `jev_sin_contexto`: los campos de la transacción, con el significado de los pocos que se conocen (monto, país, hora, score).
- `jev_con_contexto`: además, las tasas de fraude históricas que se encontraron en la exploración (`o`, `n`, `p`, hora, score, variables de historial, monto).

La API admite **30 llamadas por minuto**, así que se evalúa una **muestra estratificada de 1.500 transacciones** (75 fraudes) y se compara contra LightGBM tuneado (predicciones out-of-fold de los folds nuevos) y contra `score` **sobre las mismas filas**. El umbral de cada modelo se elige sobre la muestra, con la misma regla para todos.

- La primera ejecución tarda **~1 h 50 min**. Las respuestas se guardan en `data/interim/jev/`, así que al volver a ejecutar no se repiten las llamadas.
- Los datos se envían **sin Zero Data Retention** (no está disponible en el plan Hobby de Vercel).
- Las tasas de la variante con contexto salen de la exploración sobre todo el dataset, incluidas las filas de la muestra. Es una filtración leve, porque son tasas agregadas sobre 150.000 transacciones, y a lo sumo favorece a Jev.

In [ ]:
# Muestra estratificada: mantiene el 5% de fraude
muestra_jev = datos.groupby('fraude', group_keys=False).sample(frac=TAMANO_MUESTRA_JEV / len(datos), random_state=SEMILLA)
columnas_jev = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'k', 'l', 'm', 'n', 'o', 'p', 'monto', 'score', 'hora']
muestra_jev['fraude'].value_counts()

In [ ]:
# Instrucciones de cada variante: solo el significado de los campos conocidos, o además las tasas históricas de fraude
DESCRIPCION_CAMPOS = (
    'Estimate whether this e-commerce transaction is fraudulent. Most field names are anonymized: '
    "'monto' is the transaction amount, 'g' is the country code, 'hora' is the hour of the day (0-23) "
    "and 'score' is an external risk score from 0 to 100; the meaning of the other fields is unknown."
)
TASAS_HISTORICAS = (
    ' Historical fraud patterns in this data (overall fraud rate 5%): '
    "o = 'N' has 22% fraud, o = 'Y' 6.5% and a missing o only 2%; "
    "n = 0 has 16% fraud versus 4% for n = 1; p = 'N' has 8% versus 3% for p = 'Y'; "
    'hours 0 to 4 have 10% to 17% fraud; score between 25 and 64 is very safe (under 1.5%), '
    'while score above 80 is riskier (up to 32% above 95); low values of f, l and m are riskier; '
    'd = 50 halves the risk; amounts above 100 are riskier.'
)
instrucciones_jev = {
    'jev_sin_contexto': DESCRIPCION_CAMPOS,
    'jev_con_contexto': DESCRIPCION_CAMPOS + TASAS_HISTORICAS,
}

In [ ]:
# Probabilidad de fraude de Jev para cada transacción de la muestra (usa las respuestas guardadas si existen)
probabilidades_muestra = {
    nombre: evaluar_transacciones(muestra_jev[columnas_jev], instrucciones, CARPETA_JEV / f'{nombre}.csv')
    for nombre, instrucciones in instrucciones_jev.items()
}
probabilidades_muestra['lightgbm_tuneado'] = pd.Series(
    resultados_validacion['lightgbm_tuneado'].probabilidad_oof[muestra_jev.index], index=muestra_jev.index
)
probabilidades_muestra['score'] = muestra_jev['score'] / 100

In [ ]:
# Métricas de cada modelo sobre la muestra, con el umbral que maximiza la ganancia en esas mismas filas
fraude_muestra = muestra_jev['fraude']
monto_muestra = muestra_jev['monto']

filas = []
for nombre, probabilidad in probabilidades_muestra.items():
    umbral = umbral_optimo(fraude_muestra, monto_muestra, probabilidad)
    filas.append({
        'modelo': nombre,
        'auc_roc': roc_auc_score(fraude_muestra, probabilidad),
        'auc_pr': average_precision_score(fraude_muestra, probabilidad),
        **metricas_decision(fraude_muestra, monto_muestra, probabilidad < umbral),
        'umbral': umbral,
    })
comparacion_jev = pd.DataFrame(filas).set_index('modelo')

# Un run de MLflow por variante de Jev
for nombre in instrucciones_jev:
    with mlflow.start_run(run_name=nombre):
        mlflow.set_tags({'etapa': 'jev', 'modelo': 'jev'})
        mlflow.log_params({'modelo': 'jev', 'variante': nombre, 'tamano_muestra': len(muestra_jev)})
        mlflow.log_metrics(comparacion_jev.loc[nombre].to_dict())

comparacion_jev.sort_values('ganancia_pct_maxima', ascending=False).round(3)

In [ ]:
# Distribución de la probabilidad asignada por cada modelo, según la clase real
distribucion = pd.concat(
    [
        pd.DataFrame({'probabilidad': probabilidad, 'clase': fraude_muestra.map({0: 'Legítima', 1: 'Fraude'}), 'modelo': nombre})
        for nombre, probabilidad in probabilidades_muestra.items()
    ],
    ignore_index=True,
)
fig = px.box(
    distribucion,
    x='modelo',
    y='probabilidad',
    color='clase',
    color_discrete_map={'Legítima': COLOR_PRINCIPAL, 'Fraude': '#eb6834'},
    labels={'modelo': 'Modelo', 'probabilidad': 'Probabilidad de fraude asignada', 'clase': 'Clase real'},
    title='Probabilidad asignada a legítimas y fraudes (muestra de 1.500 transacciones)',
)
fig.update_layout(height=450)
fig.show()

**💡 Hallazgos**

- ⏳ *Pendiente: ejecutar esta sección (~1 h 50 min la primera vez) y completar los hallazgos con los resultados.*